# init-process-group-nccl — worked example 2: Context manager guaranteeing destroy

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `init-process-group-nccl`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Wrapping init/destroy in a `@contextlib.contextmanager` with a `try/finally` guarantees `destroy_process_group` runs even if the body raises. This prevents a leaked communicator from wedging subsequent runs after an error mid-training.

## Worked solution

We build a context manager around the group lifecycle.

1. Decorate with `@contextlib.contextmanager` so a generator with one `yield` becomes a `with`-usable manager.
2. Before the `yield`: set the rendezvous env vars and call `init_process_group`. We accept `dist_module` so tests can inject a mock instead of real torch.distributed.
3. Wrap the bare `yield` in `try/finally`. The `finally` block calls `destroy_process_group()` unconditionally — no guard — so cleanup runs on both the normal and the exception path.
4. We exercise it twice: once normally, once with an exception raised inside the body, and confirm destroy was called exactly once each time.

In [ ]:
import os
import contextlib

@contextlib.contextmanager
def dist_session(rank, world_size, port, dist_module, backend='gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)
    try:
        yield
    finally:
        dist_module.destroy_process_group()

class MockDist:
    def __init__(self):
        self.inits = 0
        self.destroys = 0
    def init_process_group(self, **kw):
        self.inits += 1
    def destroy_process_group(self):
        self.destroys += 1

m = MockDist()
with dist_session(0, 1, 29511, m):
    pass
try:
    with dist_session(0, 1, 29511, m):
        raise ValueError('boom')
except ValueError:
    pass
print('inits:', m.inits, 'destroys:', m.destroys)